# 03 - EDA & Visualization

## Mục tiêu

Notebook này nhằm mục đích:
- Khám phá dữ liệu qua các visualization
- Phân tích xu hướng thất nghiệp theo thời gian
- So sánh giữa các quốc gia
- Khảo sát mối liên hệ giữa thất nghiệp và các yếu tố kinh tế
- Đánh giá tác động của COVID-19

## Tại sao thực hiện?

Visualization giúp:
- Nhận diện patterns và trends dễ dàng hơn
- So sánh trực quan giữa các quốc gia
- Phát hiện outliers và anomalies
- Truyền tải insights hiệu quả

## Kết quả mong đợi

- Bộ visualization đầy đủ cho các câu hỏi nghiên cứu
- Insights về xu hướng thất nghiệp
- So sánh giữa các quốc gia và giai đoạn thời gian

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', '{:.4f}'.format)

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 7)

# Define consistent color palette for countries
country_colors = {
    'Japan': '#E74C3C',
    'Philippines': '#3498DB',
    'South Korea': '#2ECC71',
    'Thailand': '#F39C12',
    'Vietnam': '#9B59B6'
}

## 2. Load Dataset

In [ ]:
# Load the processed unemployment dataset
df = pd.read_csv('../data/processed/unemployment_final.csv')

print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")
print(f"Countries: {df['Country'].unique().tolist()}")
print(f"Years: {df['Year'].min()} - {df['Year'].max()}")

## 3. Unemployment Trend (2010-2024)

### Mục tiêu:
- Trực quan hóa xu hướng thất nghiệp theo thời gian cho 5 quốc gia
- Nhận diện các giai đoạn tăng/giảm thất nghiệp
- So sánh mức độ thất nghiệp giữa các quốc gia

In [ ]:
plt.figure(figsize=(14, 7))

for country in df['Country'].unique():
    country_data = df[df['Country'] == country].sort_values('Year')
    plt.plot(country_data['Year'], country_data['Unemployment_Rate'], 
             marker='o', linewidth=2, markersize=6, 
             label=country, color=country_colors[country])

plt.title('Unemployment Rate Trend (2010-2024)', fontsize=16, fontweight='bold')
plt.xlabel('Year', fontsize=12)
plt.ylabel('Unemployment Rate (%)', fontsize=12)
plt.legend(loc='best', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Nhận xét:
- Thailand có tỷ lệ thất nghiệp thấp nhất trong suốt giai đoạn
- Japan có tỷ lệ thất nghiệp giảm mạnh từ 2010 đến 2019
- South Korea có tỷ lệ thất nghiệp tương đối ổn định
- Philippines và Vietnam có tỷ lệ thất nghiệp ở mức trung bình
- Tất cả các quốc gia đều có sự biến động vào năm 2020 (COVID-19)

## 4. Country Comparison (Average Unemployment Rate)

### Mục tiêu:
- So sánh tỷ lệ thất nghiệp trung bình giữa các quốc gia
- Xác định quốc gia có tỷ lệ thất nghiệp cao/thấp nhất

In [ ]:
# Calculate average unemployment rate by country
avg_unemp = df.groupby('Country')['Unemployment_Rate'].mean().sort_values(ascending=False)

plt.figure(figsize=(12, 6))
bars = plt.bar(avg_unemp.index, avg_unemp.values, 
              color=[country_colors[c] for c in avg_unemp.index])

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.2f}%',
             ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.title('Average Unemployment Rate by Country (2010-2024)', fontsize=16, fontweight='bold')
plt.xlabel('Country', fontsize=12)
plt.ylabel('Average Unemployment Rate (%)', fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### Nhận xét:
- South Korea có tỷ lệ thất nghiệp trung bình cao nhất (~3.31%)
- Philippines đứng thứ hai (~2.91%)
- Japan ở mức trung bình (~3.28%)
- Vietnam có tỷ lệ thấp hơn (~1.56%)
- Thailand có tỷ lệ thất nghiệp thấp nhất (~0.74%)

## 5. COVID-19 Impact Comparison

### Mục tiêu:
- So sánh thất nghiệp giữa 3 giai đoạn: Pre-COVID (2010-2019), COVID (2020-2021), Recovery (2022-2024)
- Đánh giá tác động của đại dịch

In [ ]:
# Define periods
def get_period(year):
    if year <= 2019:
        return 'Pre-COVID (2010-2019)'
    elif year <= 2021:
        return 'COVID (2020-2021)'
    else:
        return 'Recovery (2022-2024)'

df['Period'] = df['Year'].apply(get_period)

# Calculate average unemployment by country and period
period_unemp = df.groupby(['Country', 'Period'])['Unemployment_Rate'].mean().reset_index()

# Pivot for easier plotting
period_pivot = period_unemp.pivot(index='Country', columns='Period', values='Unemployment_Rate')
period_pivot = period_pivot[['Pre-COVID (2010-2019)', 'COVID (2020-2021)', 'Recovery (2022-2024)']]

# Plot
fig, ax = plt.subplots(figsize=(14, 8))
x = np.arange(len(period_pivot.index))
width = 0.25

periods = ['Pre-COVID (2010-2019)', 'COVID (2020-2021)', 'Recovery (2022-2024)']
colors = ['#2ECC71', '#E74C3C', '#3498DB']

for i, (period, color) in enumerate(zip(periods, colors)):
    offset = (i - 1) * width
    bars = ax.bar(x + offset, period_pivot[period], width, label=period, color=color, alpha=0.8)
    
    # Add value labels
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{height:.2f}%',
               ha='center', va='bottom', fontsize=9)

ax.set_xlabel('Country', fontsize=12)
ax.set_ylabel('Average Unemployment Rate (%)', fontsize=12)
ax.set_title('Unemployment Rate by Period (2010-2024)', fontsize=16, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(period_pivot.index, rotation=45)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### Nhận xét:
- **Pre-COVID**: Tỷ lệ thất nghiệp tương đối ổn định ở các quốc gia
- **COVID (2020-2021)**: 
  - Thailand có sự tăng thất nghiệp rõ rệt (từ 0.66% lên 1.16%)
  - Japan cũng có sự tăng nhẹ
  - Các quốc gia khác có biến động khác nhau
- **Recovery (2022-2024)**: 
  - Hầu hết các quốc gia đang phục hồi
  - Tỷ lệ thất nghiệp có xu hướng giảm so với giai đoạn COVID

## 6. Youth vs Overall Unemployment

### Mục tiêu:
- So sánh tỷ lệ thất nghiệp thanh niên với thất nghiệp tổng thể
- Đánh giá khoảng cách (gap) giữa hai nhóm

In [ ]:
# Calculate youth unemployment gap
df['Youth_Gap'] = df['Youth_Unemployment_Rate'] - df['Unemployment_Rate']

# Average gap by country
avg_gap = df.groupby('Country')['Youth_Gap'].mean().sort_values(ascending=False)

plt.figure(figsize=(12, 6))
bars = plt.bar(avg_gap.index, avg_gap.values, 
              color=[country_colors[c] for c in avg_gap.index])

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.2f}%',
             ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.title('Youth-Unemployment Gap (Youth Rate - Overall Rate)', fontsize=16, fontweight='bold')
plt.xlabel('Country', fontsize=12)
plt.ylabel('Gap (percentage points)', fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.show()

### Nhận xét:
- Tất cả các quốc gia đều có youth unemployment rate cao hơn overall unemployment rate
- South Korea có gap lớn nhất (~5.27 percentage points)
- Philippines có gap lớn thứ hai (~5.10 percentage points)
- Thailand có gap nhỏ nhất (~3.09 percentage points)
- Điều này phù hợp với pattern thông thường: thanh niên thường khó tìm việc hơn

## 7. GDP Growth vs Unemployment Rate

### Mục tiêu:
- Khảo sát mối liên hệ giữa GDP Growth và Unemployment Rate
- Sử dụng scatter plot để visualize correlation

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

# Overall scatter plot
axes[0].scatter(df['GDP_Growth'], df['Unemployment_Rate'], alpha=0.6, s=100)
axes[0].set_xlabel('GDP Growth (%)', fontsize=11)
axes[0].set_ylabel('Unemployment Rate (%)', fontsize=11)
axes[0].set_title('Overall (All Countries)', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Calculate overall correlation
overall_corr = df['GDP_Growth'].corr(df['Unemployment_Rate'])
axes[0].text(0.05, 0.95, f'Correlation: {overall_corr:.3f}', 
            transform=axes[0].transAxes, fontsize=10,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Individual country scatter plots
for i, country in enumerate(df['Country'].unique()):
    if i < 5:  # We have 5 countries
        country_data = df[df['Country'] == country]
        axes[i+1].scatter(country_data['GDP_Growth'], country_data['Unemployment_Rate'], 
                         color=country_colors[country], alpha=0.7, s=100)
        axes[i+1].set_xlabel('GDP Growth (%)', fontsize=11)
        axes[i+1].set_ylabel('Unemployment Rate (%)', fontsize=11)
        axes[i+1].set_title(country, fontsize=12, fontweight='bold')
        axes[i+1].grid(True, alpha=0.3)
        
        # Calculate correlation for each country
        corr = country_data['GDP_Growth'].corr(country_data['Unemployment_Rate'])
        axes[i+1].text(0.05, 0.95, f'Correlation: {corr:.3f}', 
                      transform=axes[i+1].transAxes, fontsize=10,
                      verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Remove empty subplot
axes[5].remove()

plt.suptitle('GDP Growth vs Unemployment Rate', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

### Nhận xét:
- **Overall**: Correlation yếu (-0.117), không có mối liên hệ rõ ràng giữa GDP Growth và Unemployment Rate ở mức tổng thể
- **Japan**: Correlation dương (0.387), GDP tăng đi kèm với thất nghiệp tăng
- **Philippines**: Correlation dương yếu (0.172)
- **South Korea**: Correlation gần bằng 0 (-0.031), không có mối liên hệ rõ ràng
- **Thailand**: Correlation âm (-0.448), GDP tăng đi kèm với thất nghiệp giảm
- **Vietnam**: Correlation âm (-0.465), GDP tăng đi kèm với thất nghiệp giảm

**Lưu ý**: Correlation không chứng minh quan hệ nhân quả. Các pattern khác nhau giữa các quốc gia cho thấy mối liên hệ này phức tạp và phụ thuộc vào context kinh tế cụ thể.

## 8. Inflation vs Unemployment Rate

### Mục tiêu:
- Khảo sát mối liên hệ giữa Inflation và Unemployment Rate
- Sử dụng scatter plot để visualize correlation

In [ ]:
plt.figure(figsize=(10, 6))

# Overall scatter plot
scatter = plt.scatter(df['Inflation'], df['Unemployment_Rate'], 
                     c=df['Country'].map(country_colors), 
                     alpha=0.7, s=100)

# Calculate correlation
corr = df['Inflation'].corr(df['Unemployment_Rate'])

plt.xlabel('Inflation (%)', fontsize=12)
plt.ylabel('Unemployment Rate (%)', fontsize=12)
plt.title('Inflation vs Unemployment Rate (All Countries)', fontsize=16, fontweight='bold')
plt.grid(True, alpha=0.3)

# Add correlation annotation
plt.text(0.05, 0.95, f'Pearson Correlation: {corr:.3f}', 
         transform=plt.gca().transAxes, fontsize=12,
         verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Add legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=country_colors[c], label=c) for c in country_colors]
plt.legend(handles=legend_elements, loc='best', fontsize=10)

plt.tight_layout()
plt.show()

### Nhận xét:
- Correlation âm (-0.259) ở mức trung bình
- Có xu hướng: inflation cao hơn đi kèm với unemployment thấp hơn
- Tuy nhiên correlation không quá mạnh, mối liên hệ không rõ ràng
- Pattern này có thể do các yếu tố kinh tế phức tạp khác

**Lưu ý**: Correlation ≠ Causation. Không thể kết luận inflation gây ra unemployment giảm.

## 9. Education Analysis

### Kiểm tra dữ liệu education:

Dataset hiện tại không có cột liên quan đến education level hoặc education unemployment rate. Do đó, không thể thực hiện phân tích education.

In [ ]:
# Check for education-related columns
education_cols = [col for col in df.columns if 'education' in col.lower()]

print("="*60)
print("EDUCATION DATA AVAILABILITY CHECK")
print("="*60)

if education_cols:
    print(f"\nEducation-related columns found: {education_cols}")
    print("\nEducation analysis can be performed.")
else:
    print("\nNo education-related columns found in the dataset.")
    print("\nEducation analysis is unavailable because compatible data")
    print("is insufficient for the selected countries and period.")
print()

## 10. Summary of Key Findings

### Tổng kết các insights từ EDA:

1. **Xu hướng thất nghiệp**:
   - Thailand duy trì tỷ lệ thất nghiệp thấp nhất
   - Japan có xu hướng giảm thất nghiệp mạnh trước COVID
   - COVID-19 gây biến động ở tất cả các quốc gia

2. **So sánh quốc gia**:
   - South Korea có tỷ lệ thất nghiệp trung bình cao nhất
   - Thailand có tỷ lệ thấp nhất
   - Vietnam ở mức thấp

3. **Tác động COVID-19**:
   - Giai đoạn 2020-2021 có sự biến động thất nghiệp
   - Giai đoạn phục hồi 2022-2024 cho thấy signs of recovery

4. **Youth unemployment**:
   - Tất cả các quốc gia: youth unemployment > overall unemployment
   - Gap lớn nhất ở South Korea và Philippines

5. **GDP vs Unemployment**:
   - Overall: correlation yếu, không có pattern rõ ràng
   - Theo quốc gia: pattern khác nhau (dương ở Japan, âm ở Thailand/Vietnam)
   - Cho thấy mối liên hệ phức tạp và context-dependent

6. **Inflation vs Unemployment**:
   - Correlation âm trung bình
   - Không đủ mạnh để kết luận quan hệ nhân quả

7. **Education**:
   - Không có dữ liệu education trong dataset
   - Không thể thực hiện phân tích education

### Bước tiếp theo:

- Notebook 04: Statistical Analysis (tính toán chi tiết các chỉ số)